---

# 🧠 Neural Networks (From Scratch)

---

## 1️⃣ What is a Neural Network?

A **Neural Network (NN)** is a machine learning model inspired by the human brain.
It learns complex patterns by stacking **multiple linear models** with **non-linear activation functions**.

> Think of it as:
>
> * Linear Regression → 1 neuron
> * Logistic Regression → 1 neuron + Sigmoid
> * Neural Network → **many neurons + many layers**

---

## 2️⃣ Basic Structure of a Neural Network

A standard **Feedforward Neural Network** has three main parts:

```
Input Layer → Hidden Layer(s) → Output Layer
```

* **Input Layer**: Features (X)
* **Hidden Layers**: Feature learning
* **Output Layer**: Prediction (ŷ)

---

## 3️⃣ Neuron (Perceptron)

Each neuron performs two steps:

### Step 1: Linear Combination

```
z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
```

### Step 2: Activation

```
a = activation(z)
```

Where:

* `w` → weights
* `b` → bias
* `activation()` → non-linear function

---

## 4️⃣ Why Do We Need Activation Functions?

Without activation functions:

* Multiple layers collapse into **one linear model**
* The network **cannot learn non-linear patterns**

### Common Activation Functions

| Function | Formula                   | Used For              |
| -------- | ------------------------- | --------------------- |
| ReLU     | `max(0, z)`               | Hidden layers         |
| Sigmoid  | `1 / (1 + e⁻ᶻ)`           | Binary classification |
| Tanh     | `(eᶻ − e⁻ᶻ) / (eᶻ + e⁻ᶻ)` | Centered output       |
| Softmax  | Probabilities sum to 1    | Multi-class output    |

---

## 5️⃣ Forward Propagation

Forward propagation is how data flows through the network.

```
Input → Linear → Activation → Linear → Activation → Output
```

Mathematically for one hidden layer:

```
Z1 = XW1 + b1
A1 = ReLU(Z1)

Z2 = A1W2 + b2
A2 = Sigmoid(Z2)
```

`A2` is the final prediction (ŷ).

---

## 6️⃣ Loss Function

The **loss function** measures how wrong the prediction is.

### Common Loss Functions

| Problem Type               | Loss                      |
| -------------------------- | ------------------------- |
| Regression                 | Mean Squared Error (MSE)  |
| Binary Classification      | Binary Cross Entropy      |
| Multi-Class Classification | Categorical Cross Entropy |

Example (Binary Cross Entropy):

```
Loss = -[ y log(ŷ) + (1 - y) log(1 - ŷ) ]
```

---

## 7️⃣ Backpropagation (Learning Process)

Backpropagation updates weights using **gradient descent**.

### Steps:

1. Compute loss
2. Calculate gradients using **chain rule**
3. Update weights

```
W = W - α * ∂Loss/∂W
b = b - α * ∂Loss/∂b
```

Where:

* `α` → learning rate

> Same gradient descent you used in Linear & Logistic Regression — just applied **layer by layer backward**.

---

## 8️⃣ Training Loop (High Level)

```
for epoch in epochs:
    Forward Propagation
    Compute Loss
    Backpropagation
    Update Weights
```

---

## 9️⃣ Types of Neural Networks

| Network      | Used For                 |
| ------------ | ------------------------ |
| ANN / MLP    | Tabular data             |
| CNN          | Images                   |
| RNN / LSTM   | Sequential / Time Series |
| Transformers | NLP, GenAI               |

---

## 🔟 What We’ll Build Next

### First Neural Network (From Scratch):

* Input → Hidden Layer (ReLU)
* Output Layer (Sigmoid)
* Binary Classification
* NumPy only (no ML libraries)

---



In [0]:
import numpy as np
import pandas as pd

In [0]:
import numpy as np

class NeuralNetwork:
    
    def __init__(self, layer_sizes, task="classification", lr=0.01, seed=42):
        """
        layer_sizes: list like [n_input, n_h1, n_h2, ..., n_output]
        task: 'classification' or 'regression'
        """
        np.random.seed(seed)
        
        self.task = task
        self.lr = lr
        self.L = len(layer_sizes) - 1  # number of layers
        self.params = {}
        
        # Initialize weights and biases
        for l in range(1, len(layer_sizes)):
            self.params[f"W{l}"] = np.random.randn(
                layer_sizes[l-1], layer_sizes[l]
            ) * 0.01
            self.params[f"b{l}"] = np.zeros((1, layer_sizes[l]))
    
    # ---------- Activations ----------
    def relu(self, Z):
        return np.maximum(0, Z)
    
    def relu_derivative(self, Z):
        return (Z > 0).astype(float)
    
    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))
    
    # ---------- Forward ----------
    def forward(self, X):
        self.cache = {"A0": X}
        
        for l in range(1, self.L):
            Z = self.cache[f"A{l-1}"] @ self.params[f"W{l}"] + self.params[f"b{l}"]
            A = self.relu(Z)
            
            self.cache[f"Z{l}"] = Z
            self.cache[f"A{l}"] = A
        
        # Output layer
        ZL = self.cache[f"A{self.L-1}"] @ self.params[f"W{self.L}"] + self.params[f"b{self.L}"]
        
        if self.task == "classification":
            AL = self.sigmoid(ZL)
        else:
            AL = ZL  # linear output
        
        self.cache[f"Z{self.L}"] = ZL
        self.cache[f"A{self.L}"] = AL
        
        return AL
    
    # ---------- Loss ----------
    def loss(self, y, y_hat):
        m = y.shape[0]
        eps = 1e-8
        
        if self.task == "classification":
            return -(1/m) * np.sum(
                y * np.log(y_hat + eps) +
                (1 - y) * np.log(1 - y_hat + eps)
            )
        else:
            return (1/m) * np.sum((y - y_hat) ** 2)
    
    # ---------- Backward ----------
    def backward(self, y):
        m = y.shape[0]
        grads = {}
        
        # Output layer gradient
        dZ = self.cache[f"A{self.L}"] - y
        
        for l in reversed(range(1, self.L + 1)):
            grads[f"dW{l}"] = (1/m) * self.cache[f"A{l-1}"].T @ dZ
            grads[f"db{l}"] = (1/m) * np.sum(dZ, axis=0, keepdims=True)
            
            if l > 1:
                dA_prev = dZ @ self.params[f"W{l}"].T
                dZ = dA_prev * self.relu_derivative(self.cache[f"Z{l-1}"])
        
        # Update parameters
        for l in range(1, self.L + 1):
            self.params[f"W{l}"] -= self.lr * grads[f"dW{l}"]
            self.params[f"b{l}"] -= self.lr * grads[f"db{l}"]
    
    # ---------- Train ----------
    def fit(self, X, y, epochs=1000, verbose=True):
        for epoch in range(epochs):
            y_hat = self.forward(X)
            loss = self.loss(y, y_hat)
            self.backward(y)
            
            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")
    
    # ---------- Predict ----------
    def predict(self, X):
        y_hat = self.forward(X)
        
        if self.task == "classification":
            return (y_hat >= 0.5).astype(int)
        else:
            return y_hat


In [0]:
X = np.array([[1,2],[2,3],[3,4],[4,5]])
y = np.array([[0],[0],[1],[1]])

nn = NeuralNetwork(
    layer_sizes=[2, 8, 4, 1],   # input → hidden1 → hidden2 → output
    task="classification",
    lr=0.1
)

nn.fit(X, y, epochs=1000)
print(nn.predict(X))


In [0]:
X = np.array([[1],[2],[3],[4]])
y = np.array([[2],[4],[6],[8]])

nn = NeuralNetwork(
    layer_sizes=[1, 10, 8, 5, 1],
    task="regression",
    lr=0.01
)

nn.fit(X, y, epochs=2000)
print(nn.predict(X))
